# 04 - Train Simplifier with FLAN-T5 Small

## Stage 5 Objective

Fine-tune `google/flan-t5-small` for legal clause simplification using `data/processed/simplification_dataset.csv`.

The model is trained with the input prefix:

```text
simplify legal text: 
```

## Outputs

- Model and tokenizer: `models/simplifier/`
- Predictions: `outputs/predictions/simplifier_predictions.csv`

## Important Note

The included sample rows are only a smoke test. Add reviewed `simple_clause` targets before using the model for evaluation or reporting.

## 1. Configure Paths and Training Settings

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_builder import assign_splits
from src.simplifier import INPUT_PREFIX, build_simplification_prompt, ensure_directory, normalize_split

MODEL_NAME = 'google/flan-t5-small'
DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'simplification_dataset.csv'
MODEL_DIR = PROJECT_ROOT / 'models' / 'simplifier'
TRAINING_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'simplifier_trainer'
PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'simplifier_predictions.csv'

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 5e-5
BATCH_SIZE = 1
SEED = 42

ensure_directory(MODEL_DIR)
ensure_directory(TRAINING_OUTPUT_DIR)
ensure_directory(PREDICTIONS_PATH.parent)

print(f'Dataset path: {DATASET_PATH}')
print(f'Model output directory: {MODEL_DIR}')
print(f'Prediction output path: {PREDICTIONS_PATH}')

## 2. Load and Validate the Simplification Dataset

In [ ]:
required_columns = ['clause_id', 'clause_text', 'simple_clause', 'split']

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATASET_PATH}')

raw_df = pd.read_csv(DATASET_PATH)
missing_columns = [column for column in required_columns if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

df = raw_df.copy()
df['clause_text'] = df['clause_text'].fillna('').astype(str).str.strip()
df['simple_clause'] = df['simple_clause'].fillna('').astype(str).str.strip()
df['split'] = df['split'].map(normalize_split)
df = df[(df['clause_text'] != '') & (df['simple_clause'] != '')].reset_index(drop=True)

if len(df) < 3:
    raise ValueError('At least 3 labeled simplification rows are required for train, validation, and test splits.')

expected_splits = {'train', 'validation', 'test'}
if set(df['split']) != expected_splits:
    print('Reassigning splits so train, validation, and test are all present.')
    split_labels = assign_splits(df.to_dict(orient='records'), seed=SEED)
    df['split'] = split_labels

print(f'Using {len(df)} labeled simplification row(s).')
display(df[['clause_id', 'split', 'clause_text', 'simple_clause']].head(10))
display(df['split'].value_counts().rename_axis('split').reset_index(name='count'))

## 3. Build Hugging Face Datasets

In [ ]:
from datasets import Dataset, DatasetDict

dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(
            df[df['split'] == split_name].reset_index(drop=True),
            preserve_index=False,
        )
        for split_name in ['train', 'validation', 'test']
    }
)

dataset_dict

## 4. Load Tokenizer and Model

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Using device: {device}')

## 5. Tokenize Inputs and Targets

In [ ]:
def tokenize_batch(batch):
    inputs = [build_simplification_prompt(text, prefix=INPUT_PREFIX) for text in batch['clause_text']]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['simple_clause'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)
columns_to_remove = [column for column in dataset_dict['train'].column_names if column not in {'input_ids', 'attention_mask', 'labels'}]
tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)

tokenized_datasets

## 6. Configure Seq2SeqTrainer

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

def make_training_args():
    common_args = dict(
        output_dir=str(TRAINING_OUTPUT_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=0.01,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        logging_steps=1,
        save_strategy='epoch',
        save_total_limit=1,
        report_to=[],
        seed=SEED,
        fp16=False,
    )
    try:
        return Seq2SeqTrainingArguments(eval_strategy='epoch', **common_args)
    except TypeError:
        return Seq2SeqTrainingArguments(evaluation_strategy='epoch', **common_args)

training_args = make_training_args()

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

try:
    trainer = Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer

## 7. Fine-Tune the Model

This smoke-test configuration uses one epoch and tiny batches. Increase data size before increasing training time.

In [ ]:
train_result = trainer.train()
train_result

## 8. Evaluate on Validation and Test Splits

In [ ]:
validation_metrics = trainer.evaluate(eval_dataset=tokenized_datasets['validation'], metric_key_prefix='validation')
test_metrics = trainer.evaluate(eval_dataset=tokenized_datasets['test'], metric_key_prefix='test')

display(pd.DataFrame([validation_metrics, test_metrics]))

## 9. Save Model and Tokenizer

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f'Saved model and tokenizer to {MODEL_DIR.relative_to(PROJECT_ROOT)}')

## 10. Run Inference Examples

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

inference_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
inference_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device)

example_df = df[df['split'] == 'test'].copy()
if example_df.empty:
    example_df = df.head(3).copy()

prompts = [build_simplification_prompt(text, prefix=INPUT_PREFIX) for text in example_df['clause_text'].tolist()]
inputs = inference_tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_LENGTH)
inputs = {key: value.to(device) for key, value in inputs.items()}

generated_ids = inference_model.generate(
    **inputs,
    max_new_tokens=MAX_TARGET_LENGTH,
    num_beams=4,
)
predictions = inference_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

example_outputs = example_df[['clause_id', 'split', 'clause_text', 'simple_clause']].copy()
example_outputs['predicted_simple_clause'] = predictions
display(example_outputs)

## 11. Generate and Save Predictions

In [ ]:
all_prompts = [build_simplification_prompt(text, prefix=INPUT_PREFIX) for text in df['clause_text'].tolist()]
all_inputs = inference_tokenizer(all_prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_LENGTH)
all_inputs = {key: value.to(device) for key, value in all_inputs.items()}

all_generated_ids = inference_model.generate(
    **all_inputs,
    max_new_tokens=MAX_TARGET_LENGTH,
    num_beams=4,
)
all_predictions = inference_tokenizer.batch_decode(all_generated_ids, skip_special_tokens=True)

predictions_df = pd.DataFrame(
    {
        'clause_id': df['clause_id'],
        'split': df['split'],
        'clause_text': df['clause_text'],
        'reference_simple_clause': df['simple_clause'],
        'predicted_simple_clause': all_predictions,
    }
)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print(f'Saved {len(predictions_df)} prediction row(s) to {PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')
display(predictions_df.head(10))